# Grid Search v2 — Results Analysis

Analyzes results from `scripts/grid_search_v2.py`. Two questions:
1. **Directional correctness**: Does trading cointegrated pairs produce IC > 0 and hit rate > 50%?
2. **Cost survivability**: Which combos survive transaction costs?

In [ ]:
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import numpy as np
from pathlib import Path

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

RESULTS_DIR = Path('../results')
RESULTS_FILE = RESULTS_DIR / 'grid_search_v2.parquet'

In [ ]:
df = pl.read_parquet(RESULTS_FILE)
print(f'Loaded {len(df):,} rows | columns: {df.columns}')
df.head(5)

In [ ]:
# Available holding horizons and formation params
print('Holding horizons (bars):', sorted(df['n_bars'].unique().to_list()))
print('Formation params in results:')
for col in ['rolling_window_days', 'min_half_life', 'max_half_life', 'p_value_threshold']:
    if col in df.columns:
        print(f'  {col}: {sorted(df[col].unique().to_list())}')

# TARGET_N_BARS: the horizon used for all combo selection and ranking.
# Set to the smallest available horizon (directional correctness at the tightest window).
# Horizons beyond max_holding_minutes are not present in results (filtered at eval time).
TARGET_N_BARS = min(df['n_bars'].unique().to_list())
print(f'\nTARGET_N_BARS = {TARGET_N_BARS} bars ({TARGET_N_BARS * 15} min at 15min timeframe)')

## 1. Directional Correctness Overview

At each holding horizon: what fraction of combos have IC > 0 (signal is predictive)?

In [ ]:
# Fraction of combos with positive IC and positive net return, by horizon
summary = (
    df.group_by('n_bars')
    .agg([
        pl.len().alias('n_combos'),
        (pl.col('mean_ic_gross') > 0).sum().alias('n_positive_ic'),
        (pl.col('mean_net_return') > 0).sum().alias('n_positive_net'),
        pl.col('mean_ic_gross').mean().alias('avg_ic_gross'),
        pl.col('mean_hit_rate').mean().alias('avg_hit_rate'),
        pl.col('mean_net_return').mean().alias('avg_net_return'),
        pl.col('total_n_obs').mean().alias('avg_n_signals'),
    ])
    .with_columns([
        (pl.col('n_positive_ic') / pl.col('n_combos') * 100).alias('pct_positive_ic'),
        (pl.col('n_positive_net') / pl.col('n_combos') * 100).alias('pct_positive_net'),
    ])
    .sort('n_bars')
)
print(summary.to_pandas().to_string(index=False))

In [ ]:
# IC distribution across all combos per horizon
horizons = sorted(df['n_bars'].unique().to_list())
fig, axes = plt.subplots(1, len(horizons), figsize=(5 * len(horizons), 4), sharey=False)
if len(horizons) == 1:
    axes = [axes]

for ax, h in zip(axes, horizons):
    vals = df.filter(pl.col('n_bars') == h)['mean_ic_gross'].drop_nulls().to_numpy()
    ax.hist(vals, bins=40, color='steelblue', alpha=0.7, edgecolor='white')
    ax.axvline(0, color='red', linewidth=1.5, linestyle='--', label='IC=0')
    ax.axvline(vals.mean(), color='orange', linewidth=1.5, linestyle='-', label=f'Mean={vals.mean():.4f}')
    ax.set_title(f'{h}-bar horizon')
    ax.set_xlabel('Mean IC (gross)')
    ax.set_ylabel('Combo count')
    ax.legend(fontsize=8)

fig.suptitle('IC Distribution Across All Combos by Holding Horizon', fontsize=13)
plt.tight_layout()
plt.show()

## 2. Top Combos at Each Horizon

In [ ]:
DISPLAY_COLS = [
    'zscore_window_days', 'z_entry', 'z_exit', 'z_stop', 'max_holding_minutes',
    'mean_ic_gross', 'weighted_mean_ic_gross', 'pooled_ic_gross',
    'ic_t_stat', 'mean_gross_return', 'mean_net_return',
    'breakeven_bps', 'mean_hit_rate', 'total_n_obs'
]

for h in sorted(df['n_bars'].unique().to_list()):
    top = (
        df.filter(pl.col('n_bars') == h)
        .sort('mean_ic_gross', descending=True)
        .head(10)
        .select([c for c in DISPLAY_COLS if c in df.columns])
    )
    print(f'\n=== Top 10 by IC (gross) — {h}-bar horizon ===')
    print(top.to_pandas().to_string(index=False))

In [ ]:
# IC curve: for each unique max_holding_minutes, show mean IC at each valid horizon.
# Also plot top-5 individual combos (by IC at TARGET_N_BARS) across their horizons.

TOP_N = 5
top_combos = (
    df.filter(pl.col('n_bars') == TARGET_N_BARS)
    .sort('mean_ic_gross', descending=True)
    .head(TOP_N)
    .select(['zscore_window_days', 'z_entry', 'z_exit', 'max_holding_minutes'])
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: IC curve for top-5 combos by TARGET_N_BARS IC
ax = axes[0]
for row in top_combos.iter_rows(named=True):
    combo_df = df.filter(
        (pl.col('zscore_window_days') == row['zscore_window_days']) &
        (pl.col('z_entry') == row['z_entry']) &
        (pl.col('z_exit') == row['z_exit']) &
        (pl.col('max_holding_minutes') == row['max_holding_minutes'])
    ).sort('n_bars').to_pandas()
    label = f"zw={row['zscore_window_days']} ze={row['z_entry']} zx={row['z_exit']} mh={row['max_holding_minutes']}"
    ax.plot(combo_df['n_bars'], combo_df['mean_ic_gross'], marker='o', label=label)
    # Mark max_holding bars on x-axis
    max_hold_bars = row['max_holding_minutes'] // 15
    ax.axvline(max_hold_bars, color='grey', linewidth=0.7, linestyle=':', alpha=0.6)

ax.axhline(0, color='red', linewidth=1, linestyle='--')
ax.set_xlabel('Holding Horizon (bars)')
ax.set_ylabel('Mean IC (gross)')
ax.set_title(f'IC vs Horizon — Top {TOP_N} Combos at {TARGET_N_BARS}-bar')
ax.legend(fontsize=7, loc='upper right')

# Right: mean IC curve averaged across all combos with same max_holding_minutes
ax = axes[1]
for mh in sorted(df['max_holding_minutes'].unique().to_list()):
    sub = df.filter(pl.col('max_holding_minutes') == mh)
    curve = (
        sub.group_by('n_bars')
        .agg(pl.col('mean_ic_gross').mean())
        .sort('n_bars')
        .to_pandas()
    )
    max_hold_bars = mh // 15
    ax.plot(curve['n_bars'], curve['mean_ic_gross'], marker='o', label=f'max_hold={mh}min')
    ax.axvline(max_hold_bars, color='grey', linewidth=0.7, linestyle=':', alpha=0.5)

ax.axhline(0, color='red', linewidth=1, linestyle='--')
ax.set_xlabel('Holding Horizon (bars)')
ax.set_ylabel('Mean IC (gross, avg over combos)')
ax.set_title('Average IC Curve by max_holding_minutes\n(dotted verticals = max_hold boundary)')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()
print('Note: dotted vertical lines mark the max_holding boundary for each curve.')

def ic_heatmap(df_h, row_param, col_param, metric='mean_ic_gross', title_suffix=''):
    """Plot mean IC heatmap for two parameters."""
    pivot = (
        df_h.group_by([row_param, col_param])
        .agg(pl.col(metric).mean())
        .sort([row_param, col_param])
        .to_pandas()
        .pivot(index=row_param, columns=col_param, values=metric)
    )
    fig, ax = plt.subplots(figsize=(8, 5))
    center = 0.0
    vmax = max(abs(pivot.values.max()), abs(pivot.values.min()))
    sns.heatmap(
        pivot, annot=True, fmt='.4f', cmap='RdYlGn',
        center=center, vmin=-vmax, vmax=vmax,
        ax=ax, linewidths=0.5
    )
    ax.set_title(f'Mean {metric} | {row_param} vs {col_param}{title_suffix}')
    plt.tight_layout()
    plt.show()

df_target = df.filter(pl.col('n_bars') == TARGET_N_BARS)

# Replace None z_stop with string for groupby
df_target = df_target.with_columns(
    pl.col('z_stop').cast(pl.Utf8).fill_null('None').alias('z_stop_str')
)

ic_heatmap(df_target.to_pandas(), 'z_entry', 'zscore_window_days',
           title_suffix=f' ({TARGET_N_BARS}-bar horizon)')
ic_heatmap(df_target.to_pandas(), 'z_entry', 'z_exit',
           title_suffix=f' ({TARGET_N_BARS}-bar horizon)')
ic_heatmap(df_target.to_pandas(), 'z_entry', 'max_holding_minutes',
           title_suffix=f' ({TARGET_N_BARS}-bar horizon)')

## 3. Parameter Sensitivity — Heatmaps

Which parameters matter most for IC? Each heatmap averages IC across all other params.

In [ ]:
# Scatter: gross return vs net return across combos at TARGET_N_BARS
plot_df = df.filter(pl.col('n_bars') == TARGET_N_BARS).to_pandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: gross vs net return
ax = axes[0]
sc = ax.scatter(
    plot_df['mean_gross_return'], plot_df['mean_net_return'],
    c=plot_df['z_entry'], cmap='viridis', alpha=0.6, s=20
)
ax.axhline(0, color='red', linewidth=1, linestyle='--')
ax.axvline(0, color='red', linewidth=1, linestyle='--')
ax.set_xlabel('Mean Gross Return')
ax.set_ylabel('Mean Net Return')
ax.set_title(f'Gross vs Net Return ({TARGET_N_BARS}-bar horizon)')
plt.colorbar(sc, ax=ax, label='z_entry')

# Right: breakeven_bps distribution (profitable combos only)
ax = axes[1]
be_vals = plot_df['breakeven_bps'].dropna()
ax.hist(be_vals, bins=30, color='teal', alpha=0.7, edgecolor='white')
ax.axvline(5.0, color='red', linewidth=1.5, linestyle='--', label='Cost = 5 bps/leg')
ax.axvline(10.0, color='orange', linewidth=1.5, linestyle='--', label='Cost = 10 bps/leg')
ax.set_xlabel('Breakeven Cost (bps/leg)')
ax.set_ylabel('Combo count')
ax.set_title('Breakeven Cost Distribution (positive gross return combos)')
ax.legend()

plt.tight_layout()
plt.show()
print(f'Profitable gross combos: {len(be_vals):,} | Surviving 5bps cost: {(be_vals > 5).sum():,}')

# Pooled IC vs mean-of-daily-IC: are they consistent?
if 'pooled_ic_gross' in df.columns and 'weighted_mean_ic_gross' in df.columns:
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(plot_df['mean_ic_gross'], plot_df['pooled_ic_gross'],
               c=plot_df['total_n_obs'], cmap='plasma', alpha=0.6, s=20)
    lim = max(abs(plot_df['mean_ic_gross'].max()), abs(plot_df['pooled_ic_gross'].max()))
    ax.plot([-lim, lim], [-lim, lim], 'r--', linewidth=1)
    ax.set_xlabel('Mean-of-daily IC (noisy for low-signal days)')
    ax.set_ylabel('Pooled IC (all days combined)')
    ax.set_title(f'Pooled IC vs Mean IC ({TARGET_N_BARS}-bar)\n(color = total signal count)')
    plt.colorbar(ax.collections[0], label='total_n_obs')
    plt.tight_layout()
    plt.show()
    corr = plot_df[['mean_ic_gross', 'pooled_ic_gross']].corr().iloc[0, 1]
    print(f'Pearson correlation (mean IC vs pooled IC): {corr:.3f}')

## 4. Gross vs Net IC — Cost Drag

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Signal count vs IC by z_entry
ax = axes[0]
for z in sorted(plot_df['z_entry'].unique()):
    sub = plot_df[plot_df['z_entry'] == z]
    ax.scatter(sub['total_n_obs'], sub['mean_ic_gross'],
               label=f'z_entry={z}', alpha=0.5, s=15)
ax.axhline(0, color='red', linewidth=1, linestyle='--')
ax.set_xlabel('Total Signal Count')
ax.set_ylabel('Mean IC (gross)')
ax.set_title(f'Signal Count vs IC by z_entry ({TARGET_N_BARS}-bar horizon)')
ax.legend(fontsize=8)

# IC t-stat distribution
ax = axes[1]
t_vals = plot_df['ic_t_stat'].dropna()
ax.hist(t_vals, bins=40, color='purple', alpha=0.7, edgecolor='white')
ax.axvline(0, color='black', linewidth=1, linestyle='-')
ax.axvline(1.96, color='green', linewidth=1.5, linestyle='--', label='t=1.96 (5% sig.)')
ax.axvline(-1.96, color='green', linewidth=1.5, linestyle='--')
ax.set_xlabel('IC t-statistic')
ax.set_ylabel('Combo count')
ax.set_title(f'IC t-statistic Distribution ({TARGET_N_BARS}-bar horizon)')
ax.legend()

plt.tight_layout()
plt.show()
pct_sig = (t_vals > 1.96).mean() * 100
print(f'Combos with IC t-stat > 1.96: {pct_sig:.1f}%')

## 5. Signal Activity

Is the strategy trading enough? Low signal count → unreliable statistics.

In [ ]:
# Best by IC (gross) at TARGET_N_BARS with at least 50 total signals
min_signals = 50
best_ic = (
    df.filter(
        (pl.col('n_bars') == TARGET_N_BARS) &
        (pl.col('total_n_obs') >= min_signals)
    )
    .sort('mean_ic_gross', descending=True)
    .head(5)
    .select([c for c in DISPLAY_COLS if c in df.columns])
)
print(f'Top 5 by mean_ic_gross at {TARGET_N_BARS}-bar horizon (>= {min_signals} signals):')
print(best_ic.to_pandas().to_string(index=False))

# Best by pooled IC (less sensitive to low-signal day noise)
if 'pooled_ic_gross' in df.columns:
    best_pooled = (
        df.filter(
            (pl.col('n_bars') == TARGET_N_BARS) &
            (pl.col('total_n_obs') >= min_signals)
        )
        .sort('pooled_ic_gross', descending=True)
        .head(5)
        .select([c for c in DISPLAY_COLS if c in df.columns])
    )
    print(f'\nTop 5 by pooled_ic_gross at {TARGET_N_BARS}-bar horizon (>= {min_signals} signals):')
    print(best_pooled.to_pandas().to_string(index=False))

# Best by mean_net_return
best_net = (
    df.filter(
        (pl.col('n_bars') == TARGET_N_BARS) &
        (pl.col('total_n_obs') >= min_signals) &
        pl.col('mean_net_return').is_not_null()
    )
    .sort('mean_net_return', descending=True)
    .head(5)
    .select([c for c in DISPLAY_COLS if c in df.columns])
)
print(f'\nTop 5 by net return at {TARGET_N_BARS}-bar horizon (>= {min_signals} signals):')
print(best_net.to_pandas().to_string(index=False))

# IC across all its valid horizons for the best combo at TARGET_N_BARS
best_row = (
    df.filter(
        (pl.col('n_bars') == TARGET_N_BARS) &
        (pl.col('total_n_obs') >= min_signals)
    )
    .sort('mean_ic_gross', descending=True)
    .head(1)
    .select(['zscore_window_days', 'z_entry', 'z_exit', 'z_stop', 'max_holding_minutes'])
    .row(0, named=True)
)
print(f'Best combo: {best_row}')

best_all_horizons = df.filter(
    (pl.col('zscore_window_days') == best_row['zscore_window_days']) &
    (pl.col('z_entry') == best_row['z_entry']) &
    (pl.col('z_exit') == best_row['z_exit']) &
    (pl.col('max_holding_minutes') == best_row['max_holding_minutes'])
).sort('n_bars').to_pandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: mean IC by horizon
ax = axes[0]
ax.bar(best_all_horizons['n_bars'].astype(str), best_all_horizons['mean_ic_gross'],
       color='steelblue', alpha=0.7)
ax.axhline(0, color='red', linewidth=1)
ax.set_xlabel('Holding Horizon (bars)')
ax.set_ylabel('Mean IC (gross)')
ax.set_title(f'IC by Horizon — Best Combo')

# Right: mean vs pooled vs weighted IC comparison across horizons
ax = axes[1]
cols_to_plot = [c for c in ['mean_ic_gross', 'pooled_ic_gross', 'weighted_mean_ic_gross']
                if c in best_all_horizons.columns]
x = np.arange(len(best_all_horizons))
width = 0.25
for i, col in enumerate(cols_to_plot):
    ax.bar(x + i * width, best_all_horizons[col], width=width, alpha=0.7, label=col)
ax.axhline(0, color='red', linewidth=1)
ax.set_xticks(x + width)
ax.set_xticklabels(best_all_horizons['n_bars'].astype(str))
ax.set_xlabel('Holding Horizon (bars)')
ax.set_ylabel('IC')
ax.set_title('IC Estimators Compared (mean vs pooled vs weighted)')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Best by IC (gross) at shortest horizon with at least 50 total signals
min_signals = 50
best_ic = (
    df.filter(
        (pl.col('n_bars') == shortest_h) &
        (pl.col('total_n_obs') >= min_signals)
    )
    .sort('mean_ic_gross', descending=True)
    .head(5)
    .select([c for c in DISPLAY_COLS if c in df.columns])
)
print(f'Top 5 by IC (gross) with >= {min_signals} signals:')
print(best_ic.to_pandas().to_string(index=False))

# Best by mean_net_return
best_net = (
    df.filter(
        (pl.col('n_bars') == shortest_h) &
        (pl.col('total_n_obs') >= min_signals) &
        pl.col('mean_net_return').is_not_null()
    )
    .sort('mean_net_return', descending=True)
    .head(5)
    .select([c for c in DISPLAY_COLS if c in df.columns])
)
print(f'\nTop 5 by net return with >= {min_signals} signals:')
print(best_net.to_pandas().to_string(index=False))

In [ ]:
# IC across horizons for the best combo
best_row = (
    df.filter(
        (pl.col('n_bars') == shortest_h) &
        (pl.col('total_n_obs') >= min_signals)
    )
    .sort('mean_ic_gross', descending=True)
    .head(1)
    .select(['zscore_window_days', 'z_entry', 'z_exit', 'z_stop', 'max_holding_minutes'])
    .row(0, named=True)
)
print(f'Best combo: {best_row}')

best_all_horizons = df.filter(
    (pl.col('zscore_window_days') == best_row['zscore_window_days']) &
    (pl.col('z_entry') == best_row['z_entry']) &
    (pl.col('z_exit') == best_row['z_exit']) &
    (pl.col('max_holding_minutes') == best_row['max_holding_minutes'])
)

fig, ax = plt.subplots(figsize=(8, 4))
data = best_all_horizons.sort('n_bars').to_pandas()
ax.bar(data['n_bars'].astype(str), data['mean_ic_gross'], color='steelblue', alpha=0.7)
ax.axhline(0, color='red', linewidth=1)
ax.set_xlabel('Holding Horizon (bars)')
ax.set_ylabel('Mean IC (gross)')
ax.set_title(f'IC by Horizon — Best Combo {best_row}')
plt.tight_layout()
plt.show()